# PAN dataset preprocessing

Builds the cleaned training, validation and test pairs that every experiment except the LLMs reads.
Runs once, on the server whose home volume the experiments use.

In [ ]:
WORKERS = 8

## Imports

In [ ]:
import hashlib
import importlib
import importlib.metadata
import json
import platform
from pathlib import Path

from avcore import data, tracking

## Configuration

In [ ]:
PROJECT_DIR = Path.cwd().parents[1]
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "dataset_preprocessing"

## Expected HTML parser

clean_text removes the HTML markup with the html.parser of the standard library and BeautifulSoup.
The parser files are the same in CPython 3.11.6 - 3.11.13; from 3.11.14 on, the rest of a text after
unterminated markup such as "</3" is removed, so the cleaned data would change.

In [ ]:
EXPECTED_PARSER_MD5 = {
    "html.parser": "4362c74d58f236f46aed5ee141bcbd2b",
    "_markupbase": "e82c1e50b3513d40ad77233af215a9e5",
}
EXPECTED_BS4_VERSION = "4.15.0"
CANARY_TEXT = "I love you </3 so much"
CANARY_CLEANED_TEXT = "I love you 3 so much"

## Python patch gate

Stops the notebook before anything is read or written if the parser or BeautifulSoup differs.

In [ ]:
def check_html_parser(expected_md5: dict[str, str]) -> dict[str, str]:
    """Check the md5 of the standard-library modules behind html.parser.

    Args:
        expected_md5: The expected md5 of each module file, by module name.

    Returns:
        The md5 found, by module name.

    Raises:
        RuntimeError: If any md5 differs from the expected one.
    """
    found_md5 = {}
    for name in expected_md5:
        module = importlib.import_module(name)
        module_bytes = Path(module.__file__).read_bytes()
        found_md5[name] = hashlib.md5(module_bytes).hexdigest()
    if found_md5 != expected_md5:
        raise RuntimeError(f"html.parser files {found_md5} differ from {expected_md5}")
    return found_md5

In [ ]:
parser_md5 = check_html_parser(EXPECTED_PARSER_MD5)
bs4_version = importlib.metadata.version("beautifulsoup4")
if bs4_version != EXPECTED_BS4_VERSION:
    raise RuntimeError(f"beautifulsoup4 {bs4_version} is not {EXPECTED_BS4_VERSION}")
canary_cleaned_text = data.clean_text(CANARY_TEXT)
if canary_cleaned_text != CANARY_CLEANED_TEXT:
    raise RuntimeError(f"clean_text gave {canary_cleaned_text!r} for {CANARY_TEXT!r}")
print(f"Python {platform.python_version()}, beautifulsoup4 {bs4_version}")
print(f"parser md5 {parser_md5}")

## Run context

The commit, the clean repository check and the sha256 of the four raw files.

In [ ]:
raw_paths = [
    RAW_DIR / data.TRAIN_PAIRS_FILE,
    RAW_DIR / data.TRAIN_TRUTH_FILE,
    RAW_DIR / data.TEST_PAIRS_FILE,
    RAW_DIR / data.TEST_TRUTH_FILE,
]

In [ ]:
context = tracking.collect_run_context(PROJECT_DIR, raw_paths)
raw_sha256 = {
    path.name: context.environment[f"data_sha256.{path.name}"] for path in raw_paths
}
print(f"commit {context.environment['commit']}")
print(f"memory limit {context.environment['cgroup_memory_max']}")
for name, digest in raw_sha256.items():
    print(f"{name} {digest[:16]}")

## Build the cleaned files

Cleans both texts of every pair and splits the training pairs into training and validation.
An existing cleaned file stops the build, so a file is never overwritten.

In [ ]:
counts = data.build_cleaned_datasets(
    raw_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
    workers=WORKERS,
)
print(counts)

## Hashes of the cleaned files

In [ ]:
def hash_file(path: Path) -> str:
    """Return the sha256 of a file.

    Args:
        path: The file.

    Returns:
        The hex digest of the file bytes.
    """
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

In [ ]:
cleaned_paths = [
    PROCESSED_DIR / data.TRAIN_CLEANED_FILE,
    PROCESSED_DIR / data.VALIDATION_CLEANED_FILE,
    PROCESSED_DIR / data.TEST_CLEANED_FILE,
]
cleaned_sha256 = {path.name: hash_file(path) for path in cleaned_paths}
for name, digest in cleaned_sha256.items():
    print(f"{name} {digest[:16]}")

## Save the facts

In [ ]:
facts = {
    "commit": context.environment["commit"],
    "started_at": context.environment["started_at"],
    "uv_lock_sha256": context.environment["uv_lock_sha256"],
    "python_version": platform.python_version(),
    "beautifulsoup4_version": bs4_version,
    "parser_md5": parser_md5,
    "workers": WORKERS,
    "counts": {
        "train": counts.train,
        "validation": counts.validation,
        "test": counts.test,
    },
    "raw_sha256": raw_sha256,
    "cleaned_sha256": cleaned_sha256,
}

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
facts_path = OUTPUT_DIR / "dataset_preprocessing_facts.json"
facts_path.write_text(json.dumps(facts, indent=2), encoding="utf-8")
print(f"saved {facts_path}")